# Notebook 34 -- Luyben Fault Localization + Snowball Headline

Key experiments:
1. Snapshot posteriors for all 12 scenarios.
2. **Headline**: Snowball scenario L10 -- SBI posterior for (alpha, eta_p).
3. Fisher information 8×8 heatmap.
4. Fault classification metrics.


In [ ]:
import sys; sys.path.insert(0, '../src')
import numpy as np
import matplotlib.pyplot as plt
import pickle
from pathlib import Path

from cstr_sbi.luyben.priors import PARAM_NAMES
from cstr_sbi.luyben.inference import sample_posterior
from cstr_sbi.luyben.summaries import compute_summary_statistics_batch
from cstr_sbi.luyben.scenarios import SCENARIO_CONFIGS
import jax.numpy as jnp

with open('../results/luyben_posterior.pkl', 'rb') as f:
    bundle = pickle.load(f)
posterior = bundle['posterior']

d = np.load('../data/luyben_observations.npz')
obs = d['x']; theta_true = d['theta']; sc_ids = d['scenario_id']; t = d['t']
summaries = np.asarray(compute_summary_statistics_batch(jnp.array(obs), jnp.array(t)))
print('Data loaded, summaries shape:', summaries.shape)


## Snapshot posteriors -- all 12 scenarios

In [ ]:
param_names_list = list(PARAM_NAMES)
n_params = 8

results = {}
for sc_id in range(1, 13):
    mask = sc_ids == sc_id
    if not mask.any(): continue
    s_sc = summaries[mask]
    theta_sc = theta_true[mask]
    samples_list = [sample_posterior(posterior, s_sc[i], n_samples=5000) for i in range(min(5, len(s_sc)))]
    results[sc_id] = {
        'samples': samples_list,
        'theta_true': theta_sc[0],
        'name': [sc.name for sc in SCENARIO_CONFIGS.values() if sc.id == sc_id][0],
    }
    means = np.mean(samples_list[0], axis=0)
    print(f'L{sc_id:02d} ({results[sc_id]["name"]}):')
    for j, pname in enumerate(param_names_list):
        print(f'  {pname:10s}: true={theta_sc[0,j]:.2f}  est={means[j]:.2f}  bias={means[j]-theta_sc[0,j]:+.3f}')


## Headline: Snowball scenario L10

In [ ]:
sc10 = results.get(10)
if sc10:
    samples10 = sc10['samples'][0]  # (5000, 8)
    theta_true10 = sc10['theta_true']

    # (alpha, eta_p) joint posterior -- the banana plot
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(samples10[:, 0], samples10[:, 4], alpha=0.05, s=3, c='C0')
    ax.axvline(theta_true10[0], c='C3', ls='--', label=f'true alpha={theta_true10[0]}')
    ax.axhline(theta_true10[4], c='C1', ls='--', label=f'true eta_p={theta_true10[4]}')
    ax.set_xlabel('alpha'); ax.set_ylabel('eta_p')
    ax.set_title('L10 Snowball: (alpha, eta_p) posterior')
    ax.legend()
    plt.tight_layout()
    plt.show()
    print('Posterior means:', dict(zip(param_names_list, np.mean(samples10, axis=0))))
else:
    print('L10 not found -- run nb31 first')


## Fisher information 8×8

In [ ]:
from cstr_sbi.luyben.physics import NOMINAL_THETA, NOMINAL_INLET, NOMINAL_CTRL_ALL
from cstr_sbi.luyben.summaries import compute_summary_statistics
from cstr_sbi.luyben.simulator import simulate_em_window, warm_start_ic, apply_sensor_layer
import jax

# Numerically compute FIM via finite differences on summary statistics
eps = 1e-3
y0_nom = warm_start_ic(NOMINAL_THETA)
n_features = 65
n_params = 8

J = np.zeros((n_features, n_params))
proc_key = jax.random.PRNGKey(0)
sens_key = jax.random.PRNGKey(1)

for j in range(n_params):
    theta_plus  = NOMINAL_THETA.at[j].add(eps)
    theta_minus = NOMINAL_THETA.at[j].add(-eps)

    _, _, obs_p = simulate_em_window(theta_plus,  NOMINAL_INLET, NOMINAL_CTRL_ALL, y0_nom, key=proc_key)
    _, _, obs_m = simulate_em_window(theta_minus, NOMINAL_INLET, NOMINAL_CTRL_ALL, y0_nom, key=proc_key)
    t_out = jnp.arange(1, obs_p.shape[0] + 1) * 1.0
    s_p = np.asarray(compute_summary_statistics(obs_p, t_out))
    s_m = np.asarray(compute_summary_statistics(obs_m, t_out))
    J[:, j] = (s_p - s_m) / (2 * eps)

sigma_obs = np.ones(n_features)
FIM = J.T @ np.diag(1.0 / sigma_obs**2) @ J

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(np.log10(np.abs(FIM) + 1e-20), cmap='viridis')
ax.set_xticks(range(n_params)); ax.set_xticklabels(list(PARAM_NAMES), rotation=45, ha='right')
ax.set_yticks(range(n_params)); ax.set_yticklabels(list(PARAM_NAMES))
plt.colorbar(im, ax=ax, label='log10(|FIM|)')
ax.set_title('Fisher Information Matrix (8x8) -- log10 scale')
plt.tight_layout()
plt.show()

print('FIM diagonal (identifiability ranking):')
for j, pname in enumerate(list(PARAM_NAMES)):
    print(f'  {pname:10s}: I = {FIM[j,j]:.3e}')
